# Build APK — Thích Cầm Rìu (Google Colab)

Notebook này tự động:
1. Cài **Java 17**, **Node.js 20**, **Android SDK** (`platform-tools`, `android-34`, `build-tools;34.0.0`).
2. Clone mã nguồn từ GitHub (`spdatdd/vibe-code-throw-axes`).
3. Chạy `npm run build:web`, tạo project Android bằng **Capacitor**, và build `./gradlew assembleDebug`.
4. Tải file `ThichCamRiu-debug.apk` về máy.

In [11]:
# @title 1. Cài đặt Java 17, Node.js 20 và Android SDK
%%bash
set -e

echo "==> Cài Java 21 & công cụ..."
apt-get update -qq
apt-get install -y -qq openjdk-21-jdk unzip wget curl git > /dev/null

echo "==> Kiểm tra / cài Node.js 20..."
if ! node -v 2>/dev/null | grep -qE '^v(20|22|24|26)\.'; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null
  apt-get install -y -qq nodejs > /dev/null
fi
node -v
java -version

echo "==> Cài Android Command Line Tools..."
export ANDROID_HOME=/opt/android-sdk
if [ ! -d "$ANDROID_HOME/cmdline-tools/latest" ]; then
  mkdir -p "$ANDROID_HOME/cmdline-tools"
  wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/cmdline-tools.zip
  unzip -q /tmp/cmdline-tools.zip -d "$ANDROID_HOME/cmdline-tools"
  mv "$ANDROID_HOME/cmdline-tools/cmdline-tools" "$ANDROID_HOME/cmdline-tools/latest"
  rm -f /tmp/cmdline-tools.zip
fi

yes | "$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager" --licenses > /dev/null || true
"$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager" "platform-tools" "platforms;android-34" "build-tools;34.0.0"
echo "==> Xong môi trường Android SDK!"

==> Cài Java 21 & công cụ...
==> Kiểm tra / cài Node.js 20...
v22.23.4
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
==> Cài Android Command Line Tools...
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[=======================================] 100% Computing updates...             

==> Xong môi trường Android SDK!


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "21.0.13.1" 2026-11-17
OpenJDK Runtime Environment (build 21.0.13.1+2-1-24.04.1-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.13.1+2-1-24.04.1-Ubuntu, mixed mode, sharing)


In [12]:
# @title 2. Clone mã nguồn dự án từ GitHub
%%bash
set -e
REPO_URL="https://github.com/spdatdd/vibe-code-throw-axes.git"
BRANCH="main"

rm -rf /content/throw-axe
git clone --depth 1 -b "$BRANCH" "$REPO_URL" /content/throw-axe
ls -la /content/throw-axe

total 92
drwxr-xr-x 7 root root  4096 Nov 26 16:13 .
drwxr-xr-x 1 root root  4096 Nov 26 16:13 ..
drwxr-xr-x 3 root root  4096 Nov 26 16:13 assets
-rw-r--r-- 1 root root  4783 Nov 26 16:13 build_apk_colab.ipynb
-rw-r--r-- 1 root root   220 Nov 26 16:13 capacitor.config.json
-rw-r--r-- 1 root root  1687 Nov 26 16:13 ghi_chu_du_an.md
drwxr-xr-x 8 root root  4096 Nov 26 16:13 .git
-rw-r--r-- 1 root root    34 Nov 26 16:13 .gitignore
-rw-r--r-- 1 root root  2774 Nov 26 16:13 index.html
-rw-r--r-- 1 root root   333 Nov 26 16:13 manifest.webmanifest
-rw-r--r-- 1 root root   239 Nov 26 16:13 Nem_Riu_3D.html
-rw-r--r-- 1 root root  2897 Nov 26 16:13 note_physic_system.md
-rw-r--r-- 1 root root   304 Nov 26 16:13 package.json
-rw-r--r-- 1 root root  3956 Nov 26 16:13 readme.md
drwxr-xr-x 2 root root  4096 Nov 26 16:13 scripts
-rw-r--r-- 1 root root  2287 Nov 26 16:13 server.js
drwxr-xr-x 2 root root  4096 Nov 26 16:13 src
-rw-r--r-- 1 root root 15480 Nov 26 16:13 styles.css
drwxr-xr-x 2 root ro

Cloning into '/content/throw-axe'...


In [13]:
# @title 3. Build gói web & đóng gói APK bằng Capacitor + Gradle
%%bash
set -e
export JAVA_TOOL_OPTIONS="-Xlog:os+container=off"
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))
if [ -d "/usr/lib/jvm/java-21-openjdk-amd64" ]; then
  export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64
fi
echo "JAVA_HOME=$JAVA_HOME"
export ANDROID_HOME=/opt/android-sdk
export ANDROID_SDK_ROOT=/opt/android-sdk
export PATH="$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools"

cd /content/throw-axe
npm install --no-audit --no-fund
npm run build:web
if [ ! -d "android" ]; then
  npx cap add android
else
  npx cap sync android
fi
python3 scripts/generate-android-icons.py

cd android
chmod +x gradlew
./gradlew assembleDebug

cp app/build/outputs/apk/debug/app-debug.apk /content/ThichCamRiu-debug.apk
ls -lh /content/ThichCamRiu-debug.apk

JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64

added 98 packages in 5s

> thich-cam-riu@1.0.0 build:web
> node scripts/prepare-web.mjs

Prepared clean web bundle in: /content/throw-axe/www
✔ Adding native android project in android in 59.79ms
✔ add in 60.85ms
✔ Copying web assets from www to android/app/src/main/assets/public in 13.72ms
✔ Creating capacitor.config.json in android/app/src/main/assets in 458.15μs
✔ copy android in 19.85ms
✔ Updating Android plugins in 699.44μs
✔ update android in 12.43ms
✔ Syncing Gradle in 228.82μs
[success] android platform added!
Follow the Developer Workflow guide to get building:
https://capacitorjs.com/docs/basics/workflow

> Configure project :app

> Configure project :capacitor-cordova-android-plugins

> Task :app:preBuild UP-TO-DATE
> Task :app:preDebugBuild UP-TO-DATE
> Task :app:mergeDebugNativeDebugMetadata NO-SOURCE
> Task :capacitor-android:preBuild UP-TO-DATE
> Task :capacitor-android:preDebugBuild UP-TO-DATE
> Task :app:javaPreCompileDebug

Picked up JAVA_TOOL_OPTIONS: -Xlog:os+container=off
Picked up JAVA_TOOL_OPTIONS: -Xlog:os+container=off

Picked up JAVA_TOOL_OPTIONS: -Xlog:os+container=off

Picked up JAVA_TOOL_OPTIONS: -Xlog:os+container=off

Note: Some input files use unchecked or unsafe operations.
Note: Recompile with -Xlint:unchecked for details.


In [14]:
# @title 4. Tải file APK về máy
from google.colab import files
files.download('/content/ThichCamRiu-debug.apk')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
from google.colab import drive; import shutil; drive.mount('/content/drive'); shutil.copy('/content/ThichCamRiu-debug.apk', '/content/drive/MyDrive/ThichCamRiu-debug.apk'); print('Saved to Google Drive: /content/drive/MyDrive/ThichCamRiu-debug.apk')

Mounted at /content/drive
Saved to Google Drive: /content/drive/MyDrive/ThichCamRiu-debug.apk
